# Customer Churn Prediction - Data Preprocessing

## Objective

In the inspection stage, I explored the structure and quality of the Telco Customer Churn dataset and identified several issues that need to be addressed before model training.

In this notebook, I will prepare the raw data for machine learning by:

- Removing irrelevant and potentially leaky features
- Correcting data types
- Handling missing values
- Preparing the cleaned data for the next stage
- Validating numerical and categorical columns
- Keeping the target variable ready for the modeling stage
- Saving the cleaned dataset for EDA and feature engineering

Each preprocessing decision will be based on the findings from the data inspection rather than applying transformations blindly.

## 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np


## 2. Load the Raw Dataset

I will start with the original dataset and perform all preprocessing on a separate DataFrame. This keeps the raw data unchanged and makes it easier to trace each transformation.

In [2]:
df = pd.read_excel("../data/raw/Telco_customer_churn.xlsx")

df.head()

,CustomerID,Count,Country,State,City,Zip Code,Lat Long,Latitude,Longitude,Gender,...,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label,Churn Value,Churn Score,CLTV,Churn Reason
0,3668-QPYBK,1,United States,California,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,Male,...,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1,86,3239,Competitor made better offer
1,9237-HQITU,1,United States,California,Los Angeles,90005,"34.059281, -118.30742",34.059281,-118.307420,Female,...,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1,67,2701,Moved
2,9305-CDSKC,1,United States,California,Los Angeles,90006,"34.048013, -118.293953",34.048013,-118.293953,Female,...,Month-to-month,Yes,Electronic check,99.65,820.5,Yes,1,86,5372,Moved
3,7892-POOKP,1,United States,California,Los Angeles,90010,"34.062125, -118.315709",34.062125,-118.315709,Female,...,Month-to-month,Yes,Electronic check,104.80,3046.05,Yes,1,84,5003,Moved
4,0280-XJGEX,1,United States,California,Los Angeles,90015,"34.039224, -118.266293",34.039224,-118.266293,Male,...,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.3,Yes,1,89,5340,Competitor had better devices


In [3]:
df.shape

(7043, 33)

## 3. Feature Selection

Before cleaning and transforming the data, I need to decide which columns should be available to the model.

Some columns are identifiers, some contain information that is not useful for prediction, and some may introduce target leakage because they describe the customer's churn after it has already happened.



### Feature Selection Summary

| Feature | Decision | Reason |
|---|---|---|
| `CustomerID` | Remove | Customer identifier, no meaningful predictive information |
| `Count` | Remove | Constant feature |
| `Country` | Remove | Only one unique value |
| `State` | Remove | Only one unique value |
| `Churn Label` | Remove | Redundant with `Churn Value` |
| `Churn Reason` | Remove | Target leakage |
| `Churn Score` | Remove | Existing churn-related score |
| `CLTV` | Remove | Potential leakage / unclear calculation |
| `Lat Long` | Remove | Redundant with `Latitude` and `Longitude` |
| `Churn Value` | Target | Binary target variable |

In [4]:
columns_to_drop = [
    "CustomerID",
    "Count",
    "Country",
    "State",
    "Churn Label",
    "Churn Reason",
    "Churn Score",
    "CLTV",
    "Lat Long"
]

df = df.drop(columns = columns_to_drop)

In [5]:
# Checking that the dropped columns are actually gone (33 - 9 = 24)
df.shape

(7043, 24)

## 4. Investigating `Total Charges`

During the inspection stage, `Total Charges` was identified as an `object` column even though it represents a numerical amount.

The inspection also revealed a small number of blank values.

Before converting the column to a numerical data type or imputing the missing values, I will investigate why these values are blank and whether they are related to another feature such as customer tenure.

In [6]:
# As we can see, 11 values are blank 
df[df["Total Charges"].astype(str).str.strip() == ""]



,City,Zip Code,Latitude,Longitude,Gender,Senior Citizen,Partner,Dependents,Tenure Months,Phone Service,...,Device Protection,Tech Support,Streaming TV,Streaming Movies,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Value
2234,San Bernardino,92408,34.084909,-117.258107,Female,No,Yes,No,0,No,...,Yes,Yes,Yes,No,Two year,Yes,Bank transfer (automatic),52.55,,0
2438,Independence,93526,36.869584,-118.189241,Male,No,No,No,0,Yes,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,20.25,,0
2568,San Mateo,94401,37.590421,-122.306467,Female,No,Yes,No,0,Yes,...,Yes,No,Yes,Yes,Two year,No,Mailed check,80.85,,0
2667,Cupertino,95014,37.306612,-122.080621,Male,No,Yes,Yes,0,Yes,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,25.75,,0
2856,Redcrest,95569,40.363446,-123.835041,Female,No,Yes,No,0,No,...,Yes,Yes,Yes,No,Two year,No,Credit card (automatic),56.05,,0
4331,Los Angeles,90029,34.089953,-118.294824,Male,No,Yes,Yes,0,Yes,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,19.85,,0
4687,Sun City,92585,33.739412,-117.173334,Male,No,Yes,Yes,0,Yes,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,25.35,,0
5104,Ben Lomond,95005,37.078873,-122.090386,Female,No,Yes,Yes,0,Yes,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,20.00,,0
5719,La Verne,91750,34.144703,-117.770299,Male,No,Yes,Yes,0,Yes,...,No internet service,No internet service,No internet service,No internet service,One year,Yes,Mailed check,19.70,,0
6772,Bell,90201,33.970343,-118.171368,Female,No,Yes,Yes,0,Yes,...,Yes,Yes,Yes,No,Two year,No,Mailed check,73.35,,0


It is highly likely that these 11 customers are brand new sign-ups who haven't received their first bill yet. To confirm this hunch, let's look at how their missing fields correlate with their tenure, billing, and churn status by checking:
* `Tenure Months` (Should be 0 if they are new)
* `Monthly Charges`
* `Total Charges` 
* `Churn Value`


In [7]:
df[df["Total Charges"].astype(str).str.strip() == ""][[
    "Tenure Months",
    "Monthly Charges",
    "Total Charges",
    "Churn Value"
]]

,Tenure Months,Monthly Charges,Total Charges,Churn Value
2234,0,52.55,,0
2438,0,20.25,,0
2568,0,80.85,,0
2667,0,25.75,,0
2856,0,56.05,,0
4331,0,19.85,,0
4687,0,25.35,,0
5104,0,20.00,,0
5719,0,19.70,,0
6772,0,73.35,,0


In [8]:
df.groupby("Tenure Months")["Total Charges"].apply(
    lambda x : x.astype(str).str.strip().eq("").sum()
)

Tenure Months
0     11
1      0
2      0
3      0
4      0
      ..
68     0
69     0
70     0
71     0
72     0
Name: Total Charges, Length: 73, dtype: int64

### Observation

All 11 blank values in `Total Charges` belong to customers with `0` tenure months. Customers with tenure greater than 0 do not have blank values in this column.

This suggests that these are not random missing values. They represent newly joined customers who have not yet accumulated any total charges.

Therefore, instead of imputing these values using the mean or median, the blank values will be treated as `0`.

### Converting `Total Charges` to Numeric

Although `Total Charges` represents a numerical amount, it is currently stored as an object because some records contain blank strings.

I will first replace the blank strings with missing values and then convert the column to a numeric data type.

In [9]:
df["Total Charges"] = (
    df["Total Charges"]
    .replace(r"^\s*$", np.nan, regex=True)
    .astype(float)
)

df.loc[df["Tenure Months"] == 0, "Total Charges"] = 0

In [10]:
print("Data type:", df["Total Charges"].dtype)
print("Missing values:", df["Total Charges"].isna().sum())

Data type: float64
Missing values: 0


## 5. Data Quality Validation

After applying the initial cleaning steps, I will verify that the dataset no longer contains unexpected missing values, duplicate rows, or incorrect data types.

This validation step helps confirm that the transformations were applied correctly before moving to the next stage.

In [11]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 24 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   City               7043 non-null   str    
 1   Zip Code           7043 non-null   int64  
 2   Latitude           7043 non-null   float64
 3   Longitude          7043 non-null   float64
 4   Gender             7043 non-null   str    
 5   Senior Citizen     7043 non-null   str    
 6   Partner            7043 non-null   str    
 7   Dependents         7043 non-null   str    
 8   Tenure Months      7043 non-null   int64  
 9   Phone Service      7043 non-null   str    
 10  Multiple Lines     7043 non-null   str    
 11  Internet Service   7043 non-null   str    
 12  Online Security    7043 non-null   str    
 13  Online Backup      7043 non-null   str    
 14  Device Protection  7043 non-null   str    
 15  Tech Support       7043 non-null   str    
 16  Streaming TV       7043 non-null   

In [12]:
df.isnull().sum()

City                 0
Zip Code             0
Latitude             0
Longitude            0
Gender               0
Senior Citizen       0
Partner              0
Dependents           0
Tenure Months        0
Phone Service        0
Multiple Lines       0
Internet Service     0
Online Security      0
Online Backup        0
Device Protection    0
Tech Support         0
Streaming TV         0
Streaming Movies     0
Contract             0
Paperless Billing    0
Payment Method       0
Monthly Charges      0
Total Charges        0
Churn Value          0
dtype: int64

In [13]:
print("Duplicate rows:", df.duplicated().sum())
print("Dataset shape:", df.shape)

Duplicate rows: 0
Dataset shape: (7043, 24)


In [14]:
numerical_cols = df.select_dtypes(include=["number"]).columns.tolist()
categorical_cols = df.select_dtypes(include=["object", "string", "category"]).columns.tolist()

print("Numerical features:")
print(numerical_cols)

print("\nCategorical features:")
print(categorical_cols)

Numerical features:
['Zip Code', 'Latitude', 'Longitude', 'Tenure Months', 'Monthly Charges', 'Total Charges', 'Churn Value']

Categorical features:
['City', 'Gender', 'Senior Citizen', 'Partner', 'Dependents', 'Phone Service', 'Multiple Lines', 'Internet Service', 'Online Security', 'Online Backup', 'Device Protection', 'Tech Support', 'Streaming TV', 'Streaming Movies', 'Contract', 'Paperless Billing', 'Payment Method']


## 6. Preprocessing Summary

The initial preprocessing stage has focused on cleaning the raw dataset without introducing unnecessary transformations.

The following steps have been completed:

- Removed identifier and constant features that provide no useful predictive information.
- Removed redundant target representations.
- Removed features that could introduce target leakage.
- Removed the combined `Lat Long` field because latitude and longitude are already available separately.
- Converted `Total Charges` from text to a numerical data type.
- Investigated the 11 blank `Total Charges` values and confirmed that they occur only for customers with zero tenure.
- Treated those zero-tenure missing charges as `0`.
- Validated the resulting data types and missing values.

The dataset is now ready for the feature engineering stage.

## 7. Save the Cleaned Dataset

The dataset has now gone through the initial cleaning stage. I will save this cleaned version separately so that the original raw dataset remains unchanged.

This cleaned dataset will be used as the starting point for exploratory analysis and feature engineering.

In [15]:
import os 
os.makedirs("../Data/processed", exist_ok=True)
df.to_csv("../data/processed/telco_cleaned.csv", index=False)
print("Cleaned dataset saved successfully.")

Cleaned dataset saved successfully.
